In [10]:
%load_ext autoreload
%autoreload 2

import sys
import copy
from pathlib import Path
import cv2
import numpy as np
import torch
# Resolve paths
PROJECT_ROOT = Path("..").resolve()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

# Import pipeline classes and utilities
from dataset import ImageDataset
from feature_matching import FeatureMatcher
from pose_estimation import PoseEstimator
from triangulation import Triangulator
from bundle_adjustment import BundleAdjuster
from utils import (
    build_intrinsic_matrix, export_ply, initialize_seed_submodel, register_submodel_cameras,
    merge_submodels, sample_point_colors, point_viewing_centers, camera_centers,
    point_quality, point_quality_mask,
)
from feature_tracks import TrackGraph
from analysis import save_sfm_run
import itertools
import open3d as o3d
import matplotlib.pyplot as plt

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
# 1. Load Dataset
dataset = ImageDataset()
print(f"Loaded {len(dataset)} images.")

# 2. Build Intrinsic Matrix K
height, width = dataset[0].shape[:2]
K = build_intrinsic_matrix(width=width, height=height)

# 3. Instantiate Pipeline Modules
matcher = FeatureMatcher()
estimator = PoseEstimator(K)
triangulator = Triangulator(K)
optimizer = BundleAdjuster(K)

Loaded 177 images.


In [12]:
# 1. Pre-extract features for the entire dataset
all_kp, all_des = {}, {}
print(f"Extracting features for all {len(dataset)} images...")
for i in range(len(dataset)):
    kp, des = matcher.extract_features(dataset[i])
    all_kp[i], all_des[i] = kp, des

# 2. Push to GPU VRAM
print("Pushing all descriptors to GPU VRAM...")
all_des_gpu = {
    i: torch.tensor(des, dtype=torch.float32, device="cuda") 
    for i, des in all_des.items()
}

# 3. Build Forward Match Graph
match_graph = {}
print("Building exhaustive Match Graph across all camera pairs...")
for i, j in itertools.combinations(range(len(dataset)), 2):
    # This automatically includes Lowe's ratio test on the GPU
    good_matches = matcher.match_and_filter_gpu(all_des_gpu[i], all_des_gpu[j])
    if len(good_matches) >= 15:
        match_graph[(i, j)] = good_matches

print(f"Match Graph complete. Found {len(match_graph)} valid overlapping image pairs.")

Extracting features for all 177 images...
Pushing all descriptors to GPU VRAM...
Building exhaustive Match Graph across all camera pairs...
Match Graph complete. Found 15549 valid overlapping image pairs.


In [13]:
# Front Facade Sub-Model
cams_front, track_front = initialize_seed_submodel(
    (14, 15), estimator, triangulator, optimizer, all_kp, all_des, match_graph, TrackGraph
)

# Back Facade Sub-Model
cams_back, track_back = initialize_seed_submodel(
    (79, 80), estimator, triangulator, optimizer, all_kp, all_des, match_graph, TrackGraph
)

# Left Side Sub-Model
cams_left, track_left = initialize_seed_submodel(
    (89, 90), estimator, triangulator, optimizer, all_kp, all_des, match_graph, TrackGraph
)

# Right Side Sub-Model
cams_right, track_right = initialize_seed_submodel(
    (24, 25), estimator, triangulator, optimizer, all_kp, all_des, match_graph, TrackGraph
)

Sub-Model (14, 15) Initialized: 1424 points | RMSE: 0.1696 px
Sub-Model (79, 80) Initialized: 1182 points | RMSE: 0.2291 px
Sub-Model (89, 90) Initialized: 544 points | RMSE: 0.1954 px
Sub-Model (24, 25) Initialized: 750 points | RMSE: 0.1804 px


In [14]:
# The building is almost symmetric (front <-> back), so the match graph contains wrong
# front<->back matches. A side model could then register back-facade photos with a mirrored
# PnP pose that passes every local check, folding the back onto the front.
# The side models therefore never see the back-facade photos.
BACK_FACADE_IMGS = set(range(30, 47)) | set(range(75, 89))
SIDE_IMGS = [i for i in range(len(dataset)) if i not in BACK_FACADE_IMGS]

# PnP needs >= 30 RANSAC inliers: rejects weak poses (lower merge errors, more multi-view points).
# 50 is too strict: small seed models can never reach it early on and stop growing.
MIN_PNP_INLIERS = 30

# --- Front Facade Sub-Model ---
cams_front, track_front = register_submodel_cameras(
    cams_front, track_front, range(len(dataset)), all_kp, all_des, match_graph, K, estimator, triangulator, optimizer,
    min_inliers=MIN_PNP_INLIERS,
)

# --- Back Facade Sub-Model ---
cams_back, track_back = register_submodel_cameras(
    cams_back, track_back, range(len(dataset)), all_kp, all_des, match_graph, K, estimator, triangulator, optimizer,
    min_inliers=MIN_PNP_INLIERS,
)

# --- Left Side Sub-Model ---
cams_left, track_left = register_submodel_cameras(
    cams_left, track_left, SIDE_IMGS, all_kp, all_des, match_graph, K, estimator, triangulator, optimizer,
    min_inliers=MIN_PNP_INLIERS,
)

# --- Right Side Sub-Model ---
cams_right, track_right = register_submodel_cameras(
    cams_right, track_right, SIDE_IMGS, all_kp, all_des, match_graph, K, estimator, triangulator, optimizer,
    min_inliers=MIN_PNP_INLIERS,
)

Local BA: 3 cams | 1530 pts | 3104 obs | RMSE 0.408 -> 0.374 px
Local BA: 3 cams | 2473 pts | 3680 obs | RMSE 0.956 -> 0.800 px
Local BA: 3 cams | 2079 pts | 4371 obs | RMSE 1.071 -> 0.956 px
Local BA: 3 cams | 3017 pts | 5200 obs | RMSE 1.094 -> 0.958 px
Local BA: 3 cams | 3521 pts | 5905 obs | RMSE 1.057 -> 0.878 px
Local BA: 3 cams | 3587 pts | 6760 obs | RMSE 0.981 -> 0.847 px
Local BA: 3 cams | 3884 pts | 6927 obs | RMSE 0.941 -> 0.803 px
Local BA: 3 cams | 4290 pts | 6751 obs | RMSE 0.954 -> 0.832 px
Local BA: 3 cams | 4310 pts | 7126 obs | RMSE 1.019 -> 0.878 px
Local BA: 3 cams | 3927 pts | 7738 obs | RMSE 1.004 -> 0.924 px
Local BA: 3 cams | 3797 pts | 7162 obs | RMSE 1.109 -> 0.985 px
Local BA: 3 cams | 3615 pts | 6391 obs | RMSE 1.151 -> 0.959 px
Local BA: 3 cams | 3350 pts | 6116 obs | RMSE 1.159 -> 1.068 px
Local BA: 3 cams | 3041 pts | 5462 obs | RMSE 1.194 -> 1.093 px
Local BA: 3 cams | 2832 pts | 4546 obs | RMSE 1.311 -> 0.958 px
Local BA: 3 cams | 2449 pts | 4252 obs |

In [15]:
# Combine the 4 sub-models into one master model (front = reference frame) via Umeyama Sim(3)
merge_stats = []   # one dict per merge step, saved for results_analysis.ipynb
master_cameras, master_track, cam_source = merge_submodels(
    {
        "front": (cams_front, track_front),
        "back":  (cams_back, track_back),
        "left":  (cams_left, track_left),
        "right": (cams_right, track_right),
    },
    reference="front",
    stats=merge_stats,
)


Merging 'left' sub-model (76 cams, 47042 pts)
  -> Umeyama on shared 3D points: 17917/36872 inliers
  -> 59 shared cameras, center error after alignment: median 0.1777, max 3.0565
  -> Solved Sim(3): Scale = 0.8691, Translation = [      -5.21        3.69       11.83]
  -> Merged sub-model points: 10170 new, 36872 fused with existing master points

Merging 'right' sub-model (48 cams, 22961 pts)
  -> Umeyama on shared 3D points: 2229/20272 inliers
  -> 39 shared cameras, center error after alignment: median 0.6274, max 2.4129
  -> Solved Sim(3): Scale = 0.9084, Translation = [       6.38        1.09        3.76]
  -> Merged sub-model points: 2689 new, 20272 fused with existing master points

Merging 'back' sub-model (43 cams, 34302 pts)
  -> Umeyama on shared 3D points: 541/1124 inliers
  -> 3 shared cameras, center error after alignment: median 0.1555, max 1.5270
  -> Solved Sim(3): Scale = 0.7044, Translation = [       3.59        3.39       11.45]
  -> Merged sub-model points: 33178 

In [16]:
# Global BA in 2 rounds: filter > 20 px -> BA, then filter > 4 px -> BA.
# The front seed camera 14 (identity pose = master origin) anchors the gauge.
master_before_ba = copy.deepcopy((master_cameras, master_track))   # snapshot for results_analysis.ipynb
master_cameras, master_track = optimizer.run_global_bundle_adjustment(
    master_cameras, master_track, all_kp,
    max_iter=200, loss_scale=1.0, fixed_cam_id=14, outlier_thresholds_px=[20.0, 4.0],
)

Outlier filter (> 20.0 px): removed 17719 obs, 5121 pts -> 83996 pts | 244441 obs remain
Global BA: 132 cams | 83996 pts | 244441 obs | RMSE 2.415 -> 2.182 px
Outlier filter (> 4.0 px): removed 22796 obs, 8757 pts -> 75239 pts | 213960 obs remain
Global BA: 132 cams | 75239 pts | 213960 obs | RMSE 1.113 -> 1.068 px


In [17]:
# 1. Points + RGB colors sampled from the images
merged_points, merged_colors = sample_point_colors(master_track, all_kp, dataset)
view_centers_all = point_viewing_centers(master_track, master_cameras)

# 2. Keep only well-constrained points (>= 3 views, >= 5 deg triangulation angle, <= 2 px error).
# Most points are seen in only 2 views; their depth error along the viewing ray goes undetected
# and makes the walls "thick", which ruins the normals and the Poisson mesh.
quality = point_quality(master_track, master_cameras, all_kp, triangulator)
good = point_quality_mask(quality, min_track_length=3, min_angle_deg=5.0, max_reproj_px=2.0)
merged_points, merged_colors, view_centers_all = merged_points[good], merged_colors[good], view_centers_all[good]

# 3. Clean Point Cloud (Remove Wild Distance Outliers)
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(merged_points)
pcd.colors = o3d.utility.Vector3dVector(merged_colors / 255.0)
clean_pcd, keep_idx = pcd.remove_statistical_outlier(nb_neighbors=50, std_ratio=2.0)
clean_points = np.asarray(clean_pcd.points)
clean_colors = (np.asarray(clean_pcd.colors) * 255).round().astype(np.uint8)
print(f"Cleaned merged point cloud: {len(merged_points)} -> {len(clean_points)} points.")

# 4. Export as colored .ply
export_ply(str(PROJECT_ROOT / "outputs" / "merged_global_ba_reconstruction.ply"), clean_points, clean_colors)

# 5. Export camera viewpoints for surface meshing (normal orientation), same point order as the .ply
view_centers = view_centers_all[keep_idx]
viewpoints_file = PROJECT_ROOT / "outputs" / "merged_global_ba_viewpoints.npz"
np.savez(
    viewpoints_file,
    view_centers=view_centers,
    camera_centers=camera_centers(master_cameras, list(master_cameras)),
)
print(f"Saved per-point camera viewpoints to {viewpoints_file}")

Point quality filter (track >= 3, angle >= 5.0 deg, reproj <= 2.0 px): keeps 21957 / 75239 points (29%) | seen in only 2 views: 68%
Cleaned merged point cloud: 21957 -> 21941 points.
Successfully exported 21941 3D points to /home/semi/projects/propyläen_3d_reconstruct/outputs/merged_global_ba_reconstruction.ply
Saved per-point camera viewpoints to /home/semi/projects/propyläen_3d_reconstruct/outputs/merged_global_ba_viewpoints.npz


In [18]:
# Save this reconstruction run for notebooks/results_analysis.ipynb (plots and statistics)
# Photo groups from inspecting the images: two loops around the building (wide: 0-51, close: 52-95),
# then low-angle and passage close-ups (96-176).
image_groups = {
    "front facade": list(range(0, 24)) + list(range(52, 69)),
    "back facade":  sorted(BACK_FACADE_IMGS),
    "side A":       list(range(24, 30)) + list(range(69, 75)),
    "side B":       list(range(47, 52)) + list(range(89, 96)),
    "close-ups":    list(range(96, len(dataset))),
}
save_sfm_run(
    PROJECT_ROOT / "outputs" / "sfm_run.pkl",
    K=K, all_kp=all_kp, match_graph=match_graph, n_images=len(dataset),
    submodels={"front": (cams_front, track_front), "back": (cams_back, track_back),
               "left": (cams_left, track_left), "right": (cams_right, track_right)},
    master_before_ba=master_before_ba, master_after_ba=(master_cameras, master_track),
    cam_source=cam_source, merge_stats=merge_stats, image_groups=image_groups,
)

Saved reconstruction run for analysis to /home/semi/projects/propyläen_3d_reconstruct/outputs/sfm_run.pkl
